<a href="https://colab.research.google.com/github/yyyaassiinn/BigData_SmartCity/blob/main/BigData_SmartCity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Tutorial: The Spark Engine — Parallelism & Intelligence**

`author: Dr Amin Karami (Associate Professor in AI)`

`Summer School 2026 - Docklands Campus, University of East London (UEL)`

`E: amin.karami@ymail.com`, `a.karami@uel.ac.uk`


`W: www.aminkarami.com`

---


## 1- Setting up the Engine

In Big Data, we don't just 'open' a file. We start a Cluster. In Google Colab, we simulate this by initializing a Spark Session that uses all available CPU cores.

In [ ]:
!pip install pyspark -q
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import multiprocessing

### SparkSession Syntax

- **SparkSession.builder** → Starts the SparkSession configuration process.

- **.appName("name")** → Sets the name of the Spark application.

- **.master("local[*]")** → Defines where Spark runs:
  - `local` → Run Spark on the local machine.
  - `*` → Use all available CPU cores.

- **.config(key, value)** → Adds Spark configuration settings.
  - `spark.sql.warehouse.dir` → Location for Spark SQL warehouse files.

- **.getOrCreate()** → Creates a new SparkSession or returns an existing one.

In [ ]:
# add your code here
spark = SparkSession.builder \
        .appName("SmartCity_Day1") \
        .master("local[*]") \
        .config("spark.sql.warehouse.dir", "/content/spark-warehouse") \
        .getOrCreate()


cores = multiprocessing.cpu_count()
print(f"✅ Spark Engine Started! Your 'Cluster' has {cores} CPU cores ready for parallel work.")

✅ Spark Engine Started! Your 'Cluster' has 2 CPU cores ready for parallel work.


## 2- Data Loading

Imagine 1 person trying to count 1,000,000 cars. It takes a long time. Spark divides the work among your CPU cores using a concept called Distributed Computing.

We are downloading 2 months of the latest 2024 NYC Taxi data. By using a consistent year, we ensure the city's IoT sensors recorded the data with a uniform schema, allowing Spark's vectorized reader to process it at maximum speed.

In [ ]:
# Download 2 months of NYC data from official NYC TLC Amazon CloudFront Servers
print("📥 Downloading massive smart city datasets...")
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet -O data1.parquet
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet -O data2.parquet

# Read all files simultaneously using the wildcard (*)
df = spark.read.parquet("data*.parquet")
print(f"✅ Successfully loaded {df.count():,} records into memory!")

📥 Downloading massive smart city datasets...
✅ Successfully loaded 5,972,150 records into memory!


##3- Partitioning: The Art of Slicing

Spark breaks data into chunks called Partitions. Let's look at basic core partitioning first, which distributes the data evenly across our workers.

In [ ]:
# add your code here
print(f"The number of partitions is: {df.rdd.getNumPartitions()}")
df_optimized = df.repartition(6)
print(f"The number of partitions now is: {df_optimized.rdd.getNumPartitions()}")

The number of partitions is: 2
The number of partitions now is: 6


## Write a Partitioned Dataset

In [ ]:
# add your code here
df_optimized.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-02-15 18:24:21|  2024-02-15 18:37:00|              0|          1.3|         1|                 N|         230|         237|           1|       12.1|  5.0|    0.5|       2.

In [ ]:
df_optimized.write \
.partitionBy("payment_type") \
.mode("overwrite") \
.csv("Pay,emtType")

## 4- Pre-Processing

Sensors break. Networks fail. Before we just start deleting data, a good Data Engineer always "profiles" the dataset to prove the errors actually exist. Let's look at the minimums, maximums, and zero-passenger trips.

In [ ]:
# add your code here
df_optimized.select("passenger_count", "trip_distance", "total_amount").describe().show()

+-------+------------------+------------------+------------------+
|summary|   passenger_count|     trip_distance|      total_amount|
+-------+------------------+------------------+------------------+
|  count|           5646378|           5972150|           5972150|
|   mean|1.3326125172632792|3.7572635583499343| 26.71232295910019|
| stddev|0.8414195866902542|  240.616598537928|23.178534235895395|
|    min|                 0|               0.0|           -1000.0|
|    max|                 9|          312722.3|            9792.0|
+-------+------------------+------------------+------------------+



Now that we have proven the existence of negative distances, ridiculous maximum fares, and empty trips, we need to clean them using our distributed engine.

In [ ]:
# add your code here
zero_pass = df_optimized.filter(col('passenger_count') == 0).count()
print(f'Found {zero_pass} trips with exactly 0 passengers')

Found 65559 trips with exactly 0 passengers


In [ ]:
#add your code here
print(f"we have now {df_optimized.count()} rows")
#add 1 trip to missing values (Null) to missing passenger (assuming solo riders
df_filled = df_optimized.fillna({"passenger_count":1})
df_clean = df_filled.filter((col("passenger_count") > 0) & (col("trip_distance") > 0.1) & (col("total_amount").between(2, 1000)))
print(f"Data pre-processing complete! we have now {df_clean.count()} clean rows")

we have now 5972150 rows
Data pre-processing complete! we have now 5695510 clean rows


## 5- The SQL Bridge

You don't have to learn Python to use Spark! We can create a "Temporary View" and write standard SQL to query our massive dataset.

In [ ]:
# add your code here

df_clean.createOrReplaceTempView("city_trip")
#run SQL query
query = spark.sql("""
                  SELECT PULocationID, ROUND(AVG(total_amount), 2) AS avg_fare
                  FROM city_trip
                  WHERE trip_distance > 5
                  GROUP BY PULocationID
                  ORDER BY avg_fare DESC
                  LIMIT 10
                  """)
query.show()

+------------+--------+
|PULocationID|avg_fare|
+------------+--------+
|          44|  354.23|
|           1|  122.69|
|         109|  111.48|
|           8|   107.6|
|         115|  106.66|
|          23|  101.47|
|         221|   97.12|
|         118|   94.55|
|         172|   89.42|
|         265|   87.71|
+------------+--------+



# In-class Test

## Question 1: Understanding the Cluster Master

In the code `SparkSession.builder.master("local[*]")`, what does the "*" symbol represent?

A) It allows Spark to access all files in the current directory.

B) It instructs Spark to use all available CPU cores on the local machine for parallel processing.

C) It acts as a wildcard to connect to any available remote Spark cluster.

D) It enables "Expert Mode," which bypasses standard error logging.

## Question 2: Efficient Data Loading

Which syntax was used in the tutorial to load multiple Parquet files into a single DataFrame simultaneously?

A) df = spark.read.parquet("data1.parquet", "data2.parquet")

B) df = spark.load("data*.parquet")

C) df = spark.read.parquet("data*.parquet")

D) df = spark.read.all("data")

## Question 3: The Purpose of Repartitioning

Why did we apply `df.repartition(cores * 4)` after loading the dataset?

A) To reduce the memory footprint of the dataset on the hard drive.

B) To ensure the data is sliced into smaller chunks (partitions) so that every CPU core has multiple tasks to work on.

C) To sort the data based on the pickup timestamp.

D) To encrypt the data before processing it in the cloud.

## Question 4: Handling Data Anomalies

During the pre-processing phase, which method was used to handle missing (Null) values in the `passenger_count` column?

A) df.dropna("passenger_count")

B) df.filter(col("passenger_count") != None)

C) df.fillna({"passenger_count": 1})

D) df.replace(0, 1)

## Question 5: Partitioning Data on Disk

In the tutorial, the method `.partitionBy("payment_type")` is chained to the df.write command. What is the primary effect of this operation when saving the dataset?

A) It splits the output data into separate physical folders (directories) on the disk for each unique payment type.

B) It sorts all the rows in memory alphabetically by payment type before saving them into one massive CSV file.

C) It automatically encrypts the resulting files using the payment type as a security key.

D) It filters the dataset to drop any rows where the payment type is Null before writing.